[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/31.%20The%20Rail-Terminal%20Scheduling%20Problem/P31-Tier-7_executed.ipynb)

---


# 31. The Rail-Terminal Scheduling Problem

## Tier 7: Distributed Intelligence and Autonomous Coordination

### Goal

Build an illustrative multi-agent simulation for rail-terminal scheduling. Train agents represent jobs that need service, crane agents offer capacity through bids, and a market maker allocates available cranes to waiting trains. Agents update their bidding strategies from observed outcomes. The notebook then summarizes system performance and examines strategy stability.

### Inputs and outputs

The simulation generates train arrivals, container workloads, deadlines, agent preferences, and crane characteristics. Its outputs are a time-stamped record of assignments and system metrics, a heuristic strategy-stability analysis, and visualizations. The generated data are illustrative; they are not operational observations or results from a real terminal.

### Scope and modeling assumptions

- Each train represents one job. Its processing duration is estimated from its number of containers using a fixed workload conversion.
- A crane can serve at most one train at a time, and a train can be assigned to at most one crane at a time.
- The market maker allocates jobs using submitted prices: among feasible bids, lower-priced bids are considered first. Confidence breaks price ties.
- Deadlines influence agent utility through completion time, but the simulation does not impose a hard deadline constraint.
- Crane capacity is recorded as an agent attribute, but the generated jobs do not include cargo-weight data; therefore capacity does not determine feasibility or processing duration in this example.
- Infrastructure agents, coalition formation, explicit communication networks, disruption scenarios, and a centralized-control benchmark are not implemented. The notebook does not claim to establish a Nash equilibrium or to demonstrate real-world performance.

### Method at a glance

The notebook defines agents and their utility functions, generates bids, allocates jobs through a market mechanism, updates agent strategies from outcomes, and analyzes the resulting simulation. This is a compact, illustrative implementation of the distributed, market-based approach—not a proof that decentralized allocation is globally optimal.

### What to look for

- How price and confidence affect the order in which bids are considered.
- How exclusive crane and train assignments prevent simultaneous double-booking.
- How repeated outcomes change bidding strategies.
- Why low strategy variability is only a stability indicator, not proof of Nash equilibrium.
- How throughput, satisfaction, and fairness describe different aspects of system performance.

## 1. Mathematical model and simulation logic

### Agents, jobs, and bids

Let $\mathcal{T}$ be the set of train jobs and $\mathcal{C}$ the set of cranes. At simulation time $t$, a feasible bid $(c,i)$ pairs an available crane $c\in\mathcal{C}$ with a waiting train $i\in\mathcal{T}$. A train is waiting when it has arrived, has not completed service, and is not already assigned. A crane is available when it is not processing a train.

For train $i$, the illustrative workload estimate is

$$
p_i = \max\left(0.5,\frac{n_i}{20}\right),
$$

where $n_i$ is its number of containers and $p_i$ is its estimated processing duration in hours. The floor avoids zero-duration jobs. A crane's bid price is

$$
b_{ci} = p_i m_c\left(1 + 0.25(1-q_c)\right),
$$

where $m_c$ is the crane's learned bid-markup factor and $q_c$ is its confidence in completing the job. The market maker sorts feasible bids by increasing price and then by decreasing confidence. It accepts a bid only if neither its crane nor its train has already been assigned in that market-clearing step. This is a greedy auction rule; it is not an optimization proof or a centralized global optimum.

### Utility and learning

An agent evaluates an outcome using its preferences for time, cost, reliability, and fairness:

$$
U(o) = w_r r(o) + w_f f(o) - w_t t(o) - w_c c(o),
$$

where $r(o)$, $f(o)$, $t(o)$, and $c(o)$ are the outcome's reliability, fairness, completion duration, and price, respectively; $w_r$, $w_f$, $w_t$, and $w_c$ are the agent's corresponding preference weights. The code implements this expression in `Agent.calculate_utility`. Following an outcome, `Agent.update_strategy` records experience and adjusts the agent's strategy parameters according to whether utility is positive or non-positive. This simple update rule is illustrative and does not guarantee convergence.

### Simulation state and transitions

The simulation state records each train's status, each crane's status, active assignments, and the current time. At each time step, completed assignments are released, feasible bids are generated, the market maker selects assignments, and system metrics are recorded. The simulation begins at time zero and advances by the chosen time step until the requested duration is reached.

If $A_t$ is the set of assignments accepted at time $t$, each accepted assignment starts at $t$ and finishes at $t+p_i$. The exclusivity checks in `MarketMaker.clear_market` ensure that each crane and train appears at most once in $A_t$. The simulation uses discrete time steps to check for arrivals and clear the market, while assignment completion times are stored as continuous-valued hours.

### Metrics and interpretation

Throughput is the cumulative number of completed jobs divided by elapsed simulation time. Satisfaction is the average logistic transform of recorded agent utilities, placing values on a bounded scale; it is a model-specific indicator, not a survey result. The fairness index is Jain's index over crane assignment counts,

$$
J(x_1,\ldots,x_{|\mathcal{C}|}) =
\frac{\left(\sum_{c\in\mathcal{C}}x_c\right)^2}
{|\mathcal{C}|\sum_{c\in\mathcal{C}}x_c^2},
$$

where $x_c$ is the number of completed jobs assigned to crane $c$. If no jobs have been completed, the code reports no fairness value. Jain's index measures balance in assignment counts; it does not measure fairness across every stakeholder or account for differences in job difficulty.

The equilibrium analysis examines recent variation in each sampled agent's bid-markup factor. Low variation may indicate a stable strategy under this simulation's experiences, but it does not establish a Nash equilibrium: that requires showing that no agent can improve its utility through a unilateral strategy change, which this notebook does not test.

In [ ]:
from collections import defaultdict, deque
from copy import deepcopy
from dataclasses import dataclass
from enum import Enum
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
class AgentType(Enum):
    """Identify the operational role represented by an agent."""

    TRAIN = "train"
    CRANE = "crane"
    INFRASTRUCTURE = "infrastructure"
    MARKET_MAKER = "market_maker"


@dataclass
class Task:
    """Describe a job's workload, deadline, priority, and location."""

    id: int
    processing_time: float
    deadline: float
    priority: int
    location: Tuple[float, float]
    requirements: Dict[str, float]


@dataclass
class Bid:
    """Record an agent's offer to perform a train job."""

    agent_id: str
    task_id: int
    price: float
    completion_time: float
    confidence: float
    timestamp: float


@dataclass
class Utility:
    """Store preference weights used to evaluate outcomes."""

    time_preference: float
    cost_preference: float
    reliability_preference: float
    fairness_preference: float


@dataclass
class Agent:
    """Provide shared utility, experience, and strategy behavior."""

    id: str
    agent_type: AgentType
    utility: Utility
    current_state: Dict
    learning_rate: float = 0.1
    memory_size: int = 100

    def __post_init__(self) -> None:
        self.experience_history = deque(maxlen=self.memory_size)
        self.strategy_params = self._initialize_strategy()

    def _initialize_strategy(self) -> Dict[str, float]:
        """Set the initial bidding and cooperation parameters."""
        return {
            "bid_markup_factor": 1.0,
            "time_urgency_factor": 1.0,
            "cooperation_factor": 0.5,
            "risk_tolerance": 0.5,
        }

    def calculate_utility(self, outcome: Dict[str, float]) -> float:
        """Evaluate an outcome using this agent's preference weights."""
        time_cost = outcome.get("completion_time", 0.0) * self.utility.time_preference
        cost_penalty = outcome.get("price", 0.0) * self.utility.cost_preference
        reliability_bonus = (
            outcome.get("reliability", 1.0)
            * self.utility.reliability_preference
        )
        fairness_bonus = (
            outcome.get("fairness", 0.5)
            * self.utility.fairness_preference
        )
        return reliability_bonus + fairness_bonus - time_cost - cost_penalty

    def update_strategy(self, outcome: Dict[str, float], utility: float) -> None:
        """Store an outcome and reinforce or reduce the current strategy."""
        self.experience_history.append(
            {
                "outcome": outcome,
                "utility": utility,
                "strategy": deepcopy(self.strategy_params),
            }
        )

        # The small multiplicative update makes learning visible without
        # treating a single outcome as a complete strategy redesign.
        adjustment = 1 + self.learning_rate * (0.1 if utility > 0 else -0.2)
        for key in self.strategy_params:
            self.strategy_params[key] *= adjustment


@dataclass
class TrainAgent(Agent):
    """Represent a train job and its generated operational attributes."""

    arrival_time: float = 0.0
    departure_deadline: float = 0.0
    containers: List[int] = None
    priority: int = 2

    def __post_init__(self) -> None:
        super().__post_init__()
        if self.containers is None:
            self.containers = []


@dataclass
class CraneAgent(Agent):
    """Represent a crane and its recorded position and capabilities."""

    position: Tuple[float, float] = (0.0, 0.0)
    capacity: float = 0.0
    reliability: float = 1.0

    def __post_init__(self) -> None:
        super().__post_init__()


class MarketMaker:
    """Clear a simple price-ranked auction with exclusive assignments."""

    def __init__(self) -> None:
        self.transaction_history: List[Dict] = []

    def clear_market(
        self,
        bids: List[Bid],
        current_time: float,
        available_cranes: set,
        waiting_trains: set,
    ) -> List[Bid]:
        """Accept feasible bids in price order, without double-booking."""
        ordered_bids = sorted(
            bids,
            key=lambda bid: (bid.price, -bid.confidence, bid.task_id, bid.agent_id),
        )
        assigned_cranes = set()
        assigned_trains = set()
        accepted_bids = []

        for bid in ordered_bids:
            if bid.timestamp != current_time:
                continue
            if bid.agent_id not in available_cranes:
                continue
            if bid.task_id not in waiting_trains:
                continue
            if bid.agent_id in assigned_cranes or bid.task_id in assigned_trains:
                continue

            accepted_bids.append(bid)
            assigned_cranes.add(bid.agent_id)
            assigned_trains.add(bid.task_id)

        self.transaction_history.extend(
            {
                "time": current_time,
                "agent_id": bid.agent_id,
                "task_id": bid.task_id,
                "price": bid.price,
                "confidence": bid.confidence,
            }
            for bid in accepted_bids
        )
        return accepted_bids


print("Agent types, data structures, and the market mechanism are ready.")

In [ ]:
class DistributedIntelligenceEcosystem:
    """Coordinate train and crane agents through repeated market clearing."""

    def __init__(
        self,
        num_trains: int = 45,
        num_cranes: int = 12,
        num_tracks: int = 8,
    ) -> None:
        self.num_trains = num_trains
        self.num_cranes = num_cranes
        self.num_tracks = num_tracks
        self.agents: List[Agent] = []
        self.trains: List[TrainAgent] = []
        self.cranes: List[CraneAgent] = []
        self.market_maker = MarketMaker()
        self.active_assignments: Dict[str, Dict] = {}
        self.completed_assignments: List[Dict] = []
        self.assignment_counts = defaultdict(int)
        self.utility_history: List[float] = []
        self.system_metrics = {
            "makespan_history": [],
            "agent_satisfaction": [],
            "throughput": [],
            "fairness_index": [],
            "equilibrium_stability": [],
        }
        self._initialize_agents()

    def _initialize_agents(self) -> None:
        """Generate the train jobs and cranes used in this simulation."""
        print("Initializing the distributed rail-terminal simulation:")
        print(f"  {self.num_trains} train agents")
        print(f"  {self.num_cranes} crane agents")
        print(f"  {self.num_tracks} infrastructure agents (not simulated)")

        for index in range(self.num_trains):
            arrival_time = float(np.random.uniform(0, 24))
            departure_deadline = arrival_time + float(np.random.uniform(2, 8))
            num_containers = int(np.random.poisson(20))
            priority = int(np.random.choice([1, 2, 3], p=[0.2, 0.6, 0.2]))
            utility = Utility(
                time_preference=float(np.random.uniform(0.5, 1.5)),
                cost_preference=float(np.random.uniform(0.3, 0.8)),
                reliability_preference=float(np.random.uniform(0.7, 1.0)),
                fairness_preference=float(np.random.uniform(0.4, 0.8)),
            )
            train = TrainAgent(
                id=f"train_{index + 1}",
                agent_type=AgentType.TRAIN,
                utility=utility,
                current_state={"status": "waiting"},
                arrival_time=arrival_time,
                departure_deadline=departure_deadline,
                containers=list(range(index * 100, index * 100 + num_containers)),
                priority=priority,
            )
            self.trains.append(train)
            self.agents.append(train)

        for index in range(self.num_cranes):
            utility = Utility(
                time_preference=float(np.random.uniform(0.3, 0.8)),
                cost_preference=float(np.random.uniform(0.7, 1.2)),
                reliability_preference=float(np.random.uniform(0.5, 0.9)),
                fairness_preference=float(np.random.uniform(0.6, 1.0)),
            )
            crane = CraneAgent(
                id=f"crane_{index + 1}",
                agent_type=AgentType.CRANE,
                utility=utility,
                current_state={"status": "idle"},
                position=(
                    float(np.random.uniform(0, 1000)),
                    float(np.random.uniform(0, 500)),
                ),
                capacity=float(np.random.uniform(40, 65)),
                reliability=float(np.random.uniform(0.85, 0.99)),
            )
            self.cranes.append(crane)
            self.agents.append(crane)

        print(f"  Total simulated agents initialized: {len(self.agents)}")

    def _complete_finished_assignments(self, current_time: float) -> None:
        """Release cranes and record outcomes for jobs that have finished."""
        finished_crane_ids = [
            crane_id
            for crane_id, assignment in self.active_assignments.items()
            if assignment["completion_time"] <= current_time
        ]
        crane_by_id = {crane.id: crane for crane in self.cranes}
        train_by_id = {train.id: train for train in self.trains}

        for crane_id in finished_crane_ids:
            assignment = self.active_assignments.pop(crane_id)
            crane = crane_by_id[crane_id]
            train = train_by_id[assignment["train_id"]]
            crane.current_state["status"] = "idle"
            train.current_state["status"] = "complete"
            self.completed_assignments.append(assignment)
            self.assignment_counts[crane_id] += 1

    def run_simulation_step(self, current_time: float) -> Dict[str, float]:
        """Complete jobs, clear the auction, and record this time step."""
        self._complete_finished_assignments(current_time)
        crane_by_id = {crane.id: crane for crane in self.cranes}
        train_by_id = {train.id: train for train in self.trains}

        available_cranes = {
            crane.id for crane in self.cranes
            if crane.current_state["status"] == "idle"
        }
        waiting_trains = {
            train.id for train in self.trains
            if train.arrival_time <= current_time
            and train.current_state["status"] == "waiting"
        }

        bids = []
        for crane_id in available_cranes:
            crane = crane_by_id[crane_id]
            for train_id in waiting_trains:
                train = train_by_id[train_id]
                processing_time = max(0.5, len(train.containers) / 20.0)
                confidence = crane.reliability
                markup = crane.strategy_params["bid_markup_factor"]
                price = processing_time * markup * (1 + 0.25 * (1 - confidence))
                bids.append(
                    Bid(
                        agent_id=crane_id,
                        task_id=train_id,
                        price=price,
                        completion_time=current_time + processing_time,
                        confidence=confidence,
                        timestamp=current_time,
                    )
                )

        accepted_bids = self.market_maker.clear_market(
            bids=bids,
            current_time=current_time,
            available_cranes=available_cranes,
            waiting_trains=waiting_trains,
        )

        for bid in accepted_bids:
            crane = crane_by_id[bid.agent_id]
            train = train_by_id[bid.task_id]
            duration = bid.completion_time - current_time
            outcome = {
                "completion_time": duration,
                "price": bid.price,
                "reliability": crane.reliability,
                "fairness": 0.5,
            }
            utility = crane.calculate_utility(outcome)
            crane.update_strategy(outcome, utility)
            self.utility_history.append(utility)
            crane.current_state["status"] = "busy"
            train.current_state["status"] = "processing"
            self.active_assignments[crane.id] = {
                "crane_id": crane.id,
                "train_id": train.id,
                "start_time": current_time,
                "completion_time": bid.completion_time,
                "price": bid.price,
                "confidence": bid.confidence,
            }

        self._complete_finished_assignments(current_time)
        metrics = self.calculate_system_metrics(current_time)
        return {
            "tasks_processed": len(accepted_bids),
            "avg_utility": float(np.mean(self.utility_history))
            if self.utility_history else 0.0,
            "metrics": metrics,
        }

    def calculate_system_metrics(self, current_time: float) -> Dict[str, float]:
        """Calculate descriptive performance measures at the current time."""
        elapsed_time = max(current_time, 1e-9)
        completed_count = len(self.completed_assignments)
        throughput = completed_count / elapsed_time

        if self.utility_history:
            scaled_utilities = np.clip(np.asarray(self.utility_history) / 10.0, -50, 50)
            satisfaction = float(np.mean(1.0 / (1.0 + np.exp(-scaled_utilities))))
        else:
            satisfaction = 0.0

        counts = np.asarray(
            [self.assignment_counts[crane.id] for crane in self.cranes],
            dtype=float,
        )
        squared_sum = float(np.sum(counts**2))
        if squared_sum == 0:
            fairness_index = 0.0
        else:
            fairness_index = float(np.sum(counts) ** 2 / (len(counts) * squared_sum))

        makespan = max(
            (assignment["completion_time"] for assignment in self.completed_assignments),
            default=0.0,
        )
        return {
            "makespan": float(makespan),
            "throughput": float(throughput),
            "agent_satisfaction": satisfaction,
            "fairness_index": fairness_index,
            "completed_jobs": float(completed_count),
        }


ecosystem = DistributedIntelligenceEcosystem(
    num_trains=45,
    num_cranes=12,
    num_tracks=8,
)


In [ ]:
def run_distributed_simulation(
    ecosystem: DistributedIntelligenceEcosystem,
    duration_hours: float = 24,
    time_step: float = 0.5,
) -> Dict:
    """Run the market-based simulation and collect time-stamped records."""
    if duration_hours <= 0:
        raise ValueError("duration_hours must be positive.")
    if time_step <= 0:
        raise ValueError("time_step must be positive.")

    print("Starting the distributed-intelligence simulation:")
    print(f"  Duration: {duration_hours} hours")
    print(f"  Time step: {time_step} hours")

    simulation_results = {
        "timeline": [],
        "metrics_history": [],
        "agent_behaviors": defaultdict(list),
        "emergent_patterns": {},
    }

    current_time = 0.0
    step_count = 0
    while current_time < duration_hours:
        step_results = ecosystem.run_simulation_step(current_time)
        metrics = step_results["metrics"]
        simulation_results["timeline"].append(
            {"time": current_time, "step": step_results}
        )
        simulation_results["metrics_history"].append(
            {"time": current_time, "metrics": metrics}
        )

        for crane in ecosystem.cranes[:3]:
            simulation_results["agent_behaviors"][crane.id].append(
                {
                    "time": current_time,
                    "bid_markup": crane.strategy_params["bid_markup_factor"],
                }
            )

        step_count += 1
        if step_count % 10 == 0:
            print(
                f"  Time {current_time:.1f} hours: "
                f"{step_results['tasks_processed']} jobs assigned; "
                f"mean recorded utility {step_results['avg_utility']:.3f}"
            )
        current_time += time_step

    # Process work that finishes by the simulation horizon, including jobs
    # whose completion time falls between the final market-clearing steps.
    ecosystem._complete_finished_assignments(duration_hours)
    final_metrics = ecosystem.calculate_system_metrics(duration_hours)
    if simulation_results["metrics_history"]:
        simulation_results["metrics_history"].append(
            {"time": duration_hours, "metrics": final_metrics}
        )

    print("Simulation completed.")
    print(f"  Time steps: {step_count}")
    print(f"  Accepted market transactions: {len(ecosystem.market_maker.transaction_history)}")
    print(f"  Jobs completed by the horizon: {len(ecosystem.completed_assignments)}")
    return simulation_results


simulation_results = run_distributed_simulation(
    ecosystem,
    duration_hours=24,
    time_step=0.5,
)


In [ ]:
def analyze_nash_equilibrium(
    results: Dict,
    ecosystem: DistributedIntelligenceEcosystem,
) -> Dict:
    """Report strategy variability as a heuristic, not an equilibrium proof."""
    print("Analyzing empirical strategy stability and system performance:")
    equilibrium_analysis = {
        "strategy_stability": {},
        "convergence_metrics": {},
        "performance_summary": {},
    }

    for agent in ecosystem.agents[:10]:
        if agent.experience_history:
            recent_strategies = [
                experience["strategy"]["bid_markup_factor"]
                for experience in list(agent.experience_history)[-20:]
            ]
            if len(recent_strategies) > 1:
                strategy_variance = float(np.var(recent_strategies))
                strategy_stability = 1.0 - min(1.0, strategy_variance)
                equilibrium_analysis["strategy_stability"][agent.id] = {
                    "final_markup": float(recent_strategies[-1]),
                    "stability": float(strategy_stability),
                    "variance": strategy_variance,
                }

    stability_records = list(equilibrium_analysis["strategy_stability"].values())
    if stability_records:
        stabilities = [record["stability"] for record in stability_records]
        average_stability = float(np.mean(stabilities))
        equilibrium_analysis["convergence_metrics"] = {
            "avg_strategy_stability": average_stability,
            "agents_with_low_variation": sum(value > 0.8 for value in stabilities),
            "total_analyzed": len(stabilities),
            "stability_threshold_met": average_stability > 0.7,
        }

    if results["metrics_history"]:
        equilibrium_analysis["performance_summary"] = results["metrics_history"][-1][
            "metrics"
        ]

    convergence = equilibrium_analysis["convergence_metrics"]
    if convergence:
        print(
            "  Mean strategy-stability indicator: "
            f"{convergence['avg_strategy_stability']:.3f}"
        )
        print(
            "  Agents with low observed variation: "
            f"{convergence['agents_with_low_variation']}/"
            f"{convergence['total_analyzed']}"
        )
        print(
            "  Heuristic stability threshold met: "
            f"{convergence['stability_threshold_met']}"
        )
        print("  This diagnostic does not establish Nash equilibrium.")

    performance = equilibrium_analysis["performance_summary"]
    if performance:
        print("  Final recorded performance indicators:")
        print(f"    Completed jobs: {performance['completed_jobs']:.0f}")
        print(f"    Throughput: {performance['throughput']:.3f} jobs per hour")
        print(f"    Satisfaction indicator: {performance['agent_satisfaction']:.3f}")
        print(f"    Crane-allocation fairness index: {performance['fairness_index']:.3f}")

    return equilibrium_analysis


equilibrium_analysis = analyze_nash_equilibrium(simulation_results, ecosystem)


In [ ]:
def visualize_distributed_intelligence(
    results: Dict,
    equilibrium: Dict,
    ecosystem: DistributedIntelligenceEcosystem,
) -> None:
    """Plot recorded performance, strategy, throughput, and workload balance."""
    history = results["metrics_history"]
    if not history:
        raise ValueError("The results contain no metric history to visualize.")

    times = [record["time"] for record in history]
    metrics = [record["metrics"] for record in history]
    figure, axes = plt.subplots(2, 2, figsize=(15, 10))

    axes[0, 0].plot(
        times,
        [item["agent_satisfaction"] for item in metrics],
        label="Satisfaction indicator",
        linewidth=2,
    )
    axes[0, 0].plot(
        times,
        [item["fairness_index"] for item in metrics],
        label="Crane-allocation fairness index",
        linewidth=2,
    )
    axes[0, 0].set_xlabel("Time (hours)")
    axes[0, 0].set_ylabel("Indicator value")
    axes[0, 0].set_title("Recorded satisfaction and allocation balance")
    axes[0, 0].set_ylim(0, 1.05)
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    for agent_id, behaviors in results["agent_behaviors"].items():
        if behaviors:
            axes[0, 1].plot(
                [item["time"] for item in behaviors],
                [item["bid_markup"] for item in behaviors],
                label=agent_id,
            )
    axes[0, 1].set_xlabel("Time (hours)")
    axes[0, 1].set_ylabel("Bid-markup factor")
    axes[0, 1].set_title("Example crane bidding strategies")
    if results["agent_behaviors"]:
        axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(
        times,
        [item["throughput"] for item in metrics],
        color="tab:orange",
        linewidth=2,
    )
    axes[1, 0].set_xlabel("Time (hours)")
    axes[1, 0].set_ylabel("Completed jobs per hour")
    axes[1, 0].set_title("Cumulative throughput")
    axes[1, 0].grid(True, alpha=0.3)

    crane_ids = [crane.id for crane in ecosystem.cranes]
    assignment_counts = [ecosystem.assignment_counts[crane_id] for crane_id in crane_ids]
    axes[1, 1].bar(crane_ids, assignment_counts, color="tab:blue", alpha=0.8)
    axes[1, 1].set_xlabel("Crane agent")
    axes[1, 1].set_ylabel("Completed jobs assigned")
    axes[1, 1].set_title("Completed workload by crane")
    axes[1, 1].tick_params(axis="x", rotation=45)
    axes[1, 1].grid(True, axis="y", alpha=0.3)

    figure.suptitle("Distributed market-based scheduling indicators", fontsize=15)
    figure.tight_layout()
    plt.show()


visualize_distributed_intelligence(
    simulation_results,
    equilibrium_analysis,
    ecosystem,
)


## 2. Interpreting the approach

### What the simulation can illustrate

The model makes decentralized coordination concrete: cranes submit bids based on workload, reliability, and learned markup; the market maker considers bids in price order; and exclusive assignment checks preserve basic resource feasibility. Recorded metrics help distinguish completed-job throughput, utility-based satisfaction, and balance in crane assignments. These measures answer different questions and should not be treated as interchangeable evidence of system quality.

The strategy-stability diagnostic describes variation in recorded bid markups. Repeated outcomes and low variation can be useful observations, but the learning rule is not an equilibrium algorithm. In particular, no unilateral-deviation test is performed, so the analysis cannot support a claim of Nash equilibrium. Likewise, the simulation has no centralized benchmark, explicit disruption experiments, or stakeholder-level fairness model; conclusions about superiority, resilience, or real-terminal performance would require those additional data and analyses.

### Strengths and limitations of this modeling tier

**Potential strengths of distributed market coordination:**

- Local agents can express different preferences through their utility weights.
- A market mechanism provides a visible rule for allocating scarce cranes among waiting jobs.
- The agent structure offers a way to represent adaptation from repeated outcomes.
- Assignment exclusivity provides a basic model of shared-resource constraints.

**Important limitations of this illustrative implementation:**

- A greedy price-ranked auction need not minimize system-wide cost or makespan.
- The simple learning rule does not guarantee stable behavior or equilibrium convergence.
- Jobs use a single workload estimate, and crane capacity is not linked to cargo requirements.
- Track conflicts, travel times, communication failures, disruptions, and coalition formation are outside the model.
- The generated simulation is not a comparison with centralized control and does not establish operational performance.

### When this modeling perspective may be useful

Distributed-agent models can be useful when studying systems with multiple decision-makers, competing preferences, and shared resources. They are especially informative when the research question concerns how a specified local allocation rule behaves under varied arrivals or preferences. To make conclusions operationally meaningful, the model would need validated data, explicit infrastructure constraints, a defined benchmark, and experiments designed around the question being studied.

### Further investigation

Natural extensions within this modeling perspective include representing track occupancy and travel times, connecting crane capacity to job requirements, testing specified disruption scenarios, and evaluating the market mechanism against a clearly defined benchmark. Such extensions require additional assumptions and data; their results should be measured rather than presumed.